# Amending dataset by measuring semantic similarity in the embedding space

**Halvor Tyseng, Jonas Timmann Mjaaland, Markus Fleten Kreutzer and Tor Ole B. Odden**
Center for Computing in Science Education and Center for Interdisciplinary Education, University of Oslo, Norway.

**Disclaimer:** Usage of the method and code below will require a citation of this comptuational essay. Citation details pertaining to this research are found in the CITATION.cff file.

MIT License

Copyright (c) 2025 halvorty

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.

# Introduction
This notebook contains the code for identifying edge-cases in our dataset, meaning cases where a text should possibly be coded differently. These edge-cases are identified by calculating the distance between all texts and choosing all pairs that have a cosine distance lower than 0.15 (corresponding to high semantic similarity) and have different codes.

### Table of contents
1. [Identifying edge-cases](#one) <br>
    1.A. [Imports and load dataset](#one_a) <br>
    1.B. [Initialize embedding model](#one_b) <br>
    1.C. [Find inconsistencies](#one_c) <br>
2. [Amending edge-cases](#two)<br>
    2.A. [Load re-evaluated dataset](#two_a)<br>
    2.B. [Finetune a pre-trained embedding model](#two_b)<br>
    2.C. [Apply amending on main dataset](#two_c)<br>

<a class="anchor" id="one"></a>
# 1. Identifying edge-cases

<a class="anchor" id="one_a"></a>
## 1.A. Imports and load dataset
The notebook starts with the regular imports and loading the dataset that includes the responses and their code.

In [1]:
# Imports
import pandas as pd
import scipy as sc
from sentence_transformers import SentenceTransformer

In [2]:
# Load the data
df = pd.read_excel('data/sources.xlsx')
df

,Response,code
0,Imprecise/faulty measuring tools,L
1,air fluctuations,L
2,Slight differences in experimental setup: tabl...,L
3,The material of ramp and ball and the coeffici...,L
4,Materials issues (ball; ramps are identical),L
...,...,...
2894,imperfections in detection screen,L
2895,differences in the set up between groups (not ...,L
2896,Cosmic ray interference,L
2897,ruler differences,L


<a class="anchor" id="one_b"></a>
## 1.B. Initialize embedding model
Next, we embed the responses using sentence transformer and then calculate the cosine distance between all pairs and put them into a matrix.

In [3]:
# Embed all the responses
model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")
embeddings = model.encode(df['Response'].tolist()).tolist()

# Calculate the cosine distance between all the responses
cosine_distance = sc.spatial.distance.cdist(embeddings, embeddings, 'cosine')

<a class="anchor" id="one_c"></a>
## 1.C. Find inconsistencies
Here we identify pairs with high similarity but different codes. 
If the cosine distance is under 0.15 and the responses have different codes, mark it as an inconsistency.

In [4]:
columns = ['response_1', 'code_1', 'index_1',
           'response_2', 'code_2', 'index_2','cosine_distance']
inconst = []

for i in range(cosine_distance.shape[0]):
    temp = []
    for j in range(cosine_distance.shape[1]):
        if (cosine_distance[i, j] < 0.15 and df['code'][j] != df['code'][i]):
            temp.append([j, cosine_distance[i, j]])

    if len(temp) > 0:
        temp = min(temp, key=lambda x: x[1])
        inconst.append([df['Response'][i], df['code'][i], int(i),
                        df['Response'][temp[0]], df['code'][temp[0]], int(temp[0]),
                        temp[1]])

Further, we save the dataframe with all identified pairs into a new dataset for amending. The dataset is saved so that file can be edited in order to fix the responses that should have a different code. This re-evaluation will be done by a qualified qualitative researcher. Here we are in fact using the current dataset as an example.

In [5]:
df_inconst = pd.DataFrame(inconst, columns=columns)
df_inconst.sort_values(by='cosine_distance', ascending=True, inplace=True)
df_inconst.to_csv('data/inconsistencies.csv', index=False)

<a class="anchor" id="two"></a>
# 2. Amending edge-cases

<a class="anchor" id="two_a"></a>
## 2.A. Load re-evaluated dataset
The dataset has been amended by a qualified qualitative researcher and we get a new dataset with the amended results. It is loaded in the code-cell below. The updated codes can be found in the columns 'updated_code1' and 'updated_code2'.

In [6]:
# Load the amended data
df_inconst_amended = pd.read_csv('data/inconsistencies_amended.csv', sep=";", index_col=0)
df_inconst_amended

,text1,code1,index1,text2,code2,index2,cosine_distance,updated_code1,updated_code2
0,Measurement Error,O,506,Measurement Error,L,40,0.000000,O,O
1,Viscosity of water,L,625,viscosity of water,O,2553,0.000000,L,L
2,measurement errors,O,607,measurement errors,L,2557,0.000000,O,O
3,Randomness,P,2588,Randomness,O,2832,0.000000,O,O
4,Friction,L,1261,Friction,O,266,0.000000,L,L
...,...,...,...,...,...,...,...,...,...
664,slit isn't perfect,L,2863,the size of the slit,O,834,0.149271,L,L
665,the slope that the ball rolls down,L,1875,The slope of the triangle the ball is sitting ...,O,1383,0.149408,L,L
666,The slope of the triangle the ball is sitting ...,O,1383,the slope that the ball rolls down,L,1875,0.149408,L,L
667,difference in method of recording measurements,L,1998,Different means of measurement,O,1147,0.149739,L,L


<a class="anchor" id="two_b"></a>
## 2.B. Analyze re-evaluated dataset
Next, we want to count the amount of pairs that are resolved by the amending.

In [7]:
# Check how many of the pairs were amended
mask = df_inconst_amended['updated_code1'] == df_inconst_amended['updated_code2']
num_amended = mask.sum()
print(num_amended)

531


It turns out that 531 out of the 669 pairs are resolved(78.9%).

The next step is to implement the changes in the original dataset 'sources'.

In [8]:
# Make a new column in the df for the updated codes
df['amended_code'] = df['code']

# Update the original dataset sources with the amended codes
for row, ind in df_inconst_amended.iterrows():
    df.loc[ind['index1'], 'amended_code'] = ind['updated_code1']
    df.loc[ind['index2'], 'amended_code'] = ind['updated_code2']

# Count how many codes were changed
num_changed = (df['code'] != df['amended_code']).sum()
print(num_changed)

153


In fact only 153 responses were reclassified, however, one response could be part of many pairs. When changing the code for a single response, many pairs with this response could be resolved, making the workload less for the qualitative researcher.

<a class="anchor" id="two_c"></a>
## 2.C. Apply amending on main dataset
Finally, the finished and amended dataset is saved for analysis.

In [9]:
# Save the amended dataset
df.drop(columns=['code'], inplace=True)
df.rename(columns={'amended_code': 'code'}, inplace=True)
df.to_excel('data/sources_amended.xlsx', index=False)